In [4]:
import numpy as np
from scipy.stats import norm

class DerivativePricing:
    """Classe dédiée à la valorisation de produits dérivés (Swaps et Options)."""

    @staticmethod
    def price_interest_rate_swap(notional: float, fixed_rate: float, floating_rate: float, tenure_years: int) -> float:
        """Valorisation simplifiée d'un Interest Rate Swap (IRS) du point de vue du payeur fixe."""
        # Différentiel de taux actualisé sur la maturité
        spread = floating_rate - fixed_rate
        val = notional * spread * tenure_years
        return val

    @staticmethod
    def price_european_option(S: float, K: float, T: float, r: float, sigma: float, option_type: str = "call") -> float:
        """Valorisation d'une option européenne via le modèle de Black-Scholes."""
        if T <= 0:
            return max(0.0, S - K) if option_type == "call" else max(0.0, K - S)

        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        if option_type == "call":
            price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

        return price

In [5]:
class MarginCalculator:
    """Calcule la Marge de Variation (VM) et la Marge Initiale (IM)."""

    @staticmethod
    def calculate_variation_margin(current_val: float, previous_val: float) -> float:
        """La Variation Margin couvre l'évolution quotidienne de la valeur de marché."""
        return current_val - previous_val

    @staticmethod
    def calculate_initial_margin(notional: float, volatility: float, confidence_level_factor: float = 2.33) -> float:
        """
        La Marge Initiale couvre le risque de mouvements de marché extrêmes
        en cas de défaut (VaR à 99% généralement, soit facteur ~2.33).
        """
        # Formule simplifiée inspirée de la VaR paramétrique (1 jour / 10 jours)
        im = notional * volatility * confidence_level_factor * (10 / 252) ** 0.5
        return im

In [1]:
import sqlite3
import numpy as np
import pandas as pd

# 1. Génération d'un jeu de données simulant les disclosures CPMI-IOSCO des CCPs
np.random.seed(42)
ccp_names = ["LCH Ltd", "Eurex Clearing", "CME Clearing", "ICE Clear", "Nasdaq Clearing"]
quarters = ["Q1-2024", "Q2-2024", "Q3-2024", "Q4-2024", "Q1-2025", "Q2-2025", "Q3-2025", "Q4-2025"]

data = []
for ccp in ccp_names:
    # Base évolutive pour simuler l'historique YoY
    base_df = np.random.uniform(5000, 15000) # Default Fund Size en Millions USD
    for q in quarters:
        df_size = base_df * np.random.normal(1.02, 0.05)
        im_held = df_size * np.random.uniform(3.5, 6.0)
        skin_in_game = df_size * np.random.uniform(0.05, 0.15)
        liquid_res = df_size * np.random.uniform(1.5, 3.0)
        peak_liq = liquid_res * np.random.uniform(0.7, 1.2)
        volatility = np.random.uniform(0.10, 0.35)

        data.append({
            "CCP_Name": ccp,
            "Quarter": q,
            "Default_Fund_Size": round(df_size, 2),
            "Initial_Margins_Held": round(im_held, 2),
            "CCP_Skin_In_The_Game": round(skin_in_game, 2),
            "Liquid_Resources": round(liquid_res, 2),
            "Peak_Liquidity_Need": round(peak_liq, 2),
            "Volatility_Index": round(volatility, 4)
        })

df_ccp = pd.DataFrame(data)

# 2. Ingestion dans une base SQLite locale (Colab)
db_conn = sqlite3.connect("ccp_risk_database.db")
df_ccp.to_sql("ccp_financials", db_conn, if_exists="replace", index=False)
db_conn.close()

print("Succès : Base de données SQLite 'ccp_risk_database.db' initialisée avec les données CCP.")
display(df_ccp.head())

Succès : Base de données SQLite 'ccp_risk_database.db' initialisée avec les données CCP.


,CCP_Name,Quarter,Default_Fund_Size,Initial_Margins_Held,CCP_Skin_In_The_Game,Liquid_Resources,Peak_Liquidity_Need,Volatility_Index
0,LCH Ltd,Q1-2024,8434.12,32808.60,470.69,23609.32,23622.49,0.2770
1,LCH Ltd,Q2-2024,9059.76,32175.37,1331.70,24902.22,20075.41,0.1455
2,LCH Ltd,Q3-2024,8670.46,41721.32,808.04,16793.33,16892.85,0.1349
3,LCH Ltd,Q4-2024,8516.24,36026.76,737.81,18600.35,20322.52,0.1499
4,LCH Ltd,Q1-2025,10039.50,36304.09,1111.92,17627.21,12912.39,0.3372


In [2]:
class CCPSmartRatingEngine:
    def __init__(self, db_path="ccp_risk_database.db"):
        self.db_path = db_path

    def fetch_latest_data(self):
        conn = sqlite3.connect(self.db_path)
        query = """
            SELECT * FROM ccp_financials
            WHERE Quarter = (SELECT MAX(Quarter) FROM ccp_financials)
        """
        df = pd.read_sql(query, conn)
        conn.close()
        return df

    def compute_ratings_and_limits(self):
        df = self.fetch_latest_data()
        results = []

        for _, row in df.iterrows():
            # Calcul des ratios clés LOD2
            lcr = row["Liquid_Resources"] / row["Peak_Liquidity_Need"]
            sitg_ratio = row["CCP_Skin_In_The_Game"] / row["Default_Fund_Size"]

            # Score composite (0 à 100) basé sur la liquidité, le skin-in-the-game et la volatilité
            score = (lcr * 40) + (sitg_ratio * 40) + ((1 - row["Volatility_Index"]) * 20)
            score = max(0, min(100, score))

            # Mapping vers un Rating interne et proxy de PD (Probability of Default)
            if score >= 80:
                rating, pd_proxy, haircut = "AAA", 0.01, 0.85
            elif score >= 65:
                rating, pd_proxy, haircut = "AA", 0.03, 0.70
            elif score >= 50:
                rating, pd_proxy, haircut = "A", 0.08, 0.55
            elif score >= 35:
                rating, pd_proxy, haircut = "BBB", 0.15, 0.40
            else:
                rating, pd_proxy, haircut = "BB- or Below", 0.30, 0.20

            # Module 2 : Limit Setting Engine (Séparation Clearing vs Bilatéral)
            clearing_limit = row["Default_Fund_Size"] * haircut
            bilateral_limit = row["Liquid_Resources"] * haircut * 0.5

            results.append({
                "CCP_Name": row["CCP_Name"],
                "Liquidity_Coverage_Ratio": round(lcr, 2),
                "Skin_In_The_Game_Ratio": round(sitg_ratio, 4),
                "Risk_Score": round(score, 2),
                "Internal_Rating": rating,
                "Implied_PD": f"{pd_proxy * 100}%",
                "Clearing_Limit_MUSD": round(clearing_limit, 2),
                "Bilateral_Limit_MUSD": round(bilateral_limit, 2)
            })

        return pd.DataFrame(results)

engine = CCPSmartRatingEngine()
df_scored = engine.compute_ratings_and_limits()
display(df_scored)

,CCP_Name,Liquidity_Coverage_Ratio,Skin_In_The_Game_Ratio,Risk_Score,Internal_Rating,Implied_PD,Clearing_Limit_MUSD,Bilateral_Limit_MUSD
0,LCH Ltd,1.26,0.1020,67.71,AA,3.0%,6194.71,7186.06
1,Eurex Clearing,1.40,0.0525,72.81,AA,3.0%,8816.53,7325.82
2,CME Clearing,1.22,0.0779,69.18,AA,3.0%,5337.51,7639.04
3,ICE Clear,1.28,0.1142,69.30,AA,3.0%,7335.45,5964.49
4,Nasdaq Clearing,1.02,0.0817,57.47,A,8.0%,6432.33,5641.92


In [3]:
class DueDiligenceGenerator:
    def __init__(self, db_path="ccp_risk_database.db"):
        self.db_path = db_path

    def generate_report(self, ccp_target):
        conn = sqlite3.connect(self.db_path)

        # Récupération de l'historique YoY pour la CCP ciblée
        query = f"SELECT * FROM ccp_financials WHERE CCP_Name = '{ccp_target}' ORDER BY Quarter ASC"
        df_hist = pd.read_sql(query, conn)
        conn.close()

        if df_hist.empty:
            return f"Aucune donnée trouvée pour la CCP : {ccp_target}"

        latest = df_hist.iloc[-1]
        previous_year = df_hist.iloc[-5] if len(df_hist) >= 5 else df_hist.iloc[0]

        # Calcul des variations YoY
        yoy_df_growth = ((latest["Default_Fund_Size"] - previous_year["Default_Fund_Size"]) / previous_year["Default_Fund_Size"]) * 100

        # Exécution du moteur de notation pour obtenir le rating actuel
        engine = CCPSmartRatingEngine(self.db_path)
        df_scored = engine.compute_ratings_and_limits()
        ccp_metrics = df_scored[df_scored["CCP_Name"] == ccp_target].iloc[0]

        # Rédaction du rapport automatisé (LOD2 Narrative)
        report = f"""
        ======================================================================
        AUTOMATED DUE DILIGENCE & ANNUAL REVIEW REPORT (LOD2)
        Institution : {ccp_target}
        Période d'analyse : {latest['Quarter']} (Comparaison YoY vs {previous_year['Quarter']})
        ======================================================================

        [1. COUNTERPARTY CREDIT RATING & SCORING]
        - Internal Rating Assigné : {ccp_metrics['Internal_Rating']}
        - Score de Risque Global   : {ccp_metrics['Risk_Score']} / 100
        - Probabilité de Défaut (PD) : {ccp_metrics['Implied_PD']}

        [2. LIMIT SETTING & EXPOSURE THRESHOLDS]
        - Limite Activité Clearing (Default Fund Sizing) : {ccp_metrics['Clearing_Limit_MUSD']} M$
        - Limite Activité Bilatérale (FMI Directe)         : {ccp_metrics['Bilateral_Limit_MUSD']} M$

        [3. QUANTITATIVE DISCLOSURES & TREND ANALYSIS (YoY)]
        - Default Fund Size Actuel : {latest['Default_Fund_Size']} M$ (Évolution YoY : {yoy_df_growth:+.2f}%)
        - Liquid Resources Available: {latest['Liquid_Resources']} M$
        - Peak Liquidity Need       : {latest['Peak_Liquidity_Need']} M$
        - Liquidity Coverage Ratio  : {ccp_metrics['Liquidity_Coverage_Ratio']}x

        [4. AUTOMATED RISK COMMENTARY & ESCALATION]
        """

        if ccp_metrics['Risk_Score'] < 50:
            report += f"\n[ALERTE ROUGE] Dégradation significative du profil de risque pour {ccp_target}. Recommandation de revue par le comité des risques de contrepartie."
        elif yoy_df_growth < -5:
            report += f"\n[AVERTISSEMENT] Baisse de la taille du fonds de garantie de {abs(yoy_df_growth):.2f}% sur un an. Surveillance renforcée requise."
        else:
            report += f"\n[STATUT STABLE] Le profil financier de {ccp_target} est robuste. Les métriques de liquidité et de cascade de défaut respectent les seuils de tolérance de la Société Générale."

        print(report)

# Exécution du rapport pour LCH Ltd
dd_gen = DueDiligenceGenerator()
dd_gen.generate_report("LCH Ltd")


        AUTOMATED DUE DILIGENCE & ANNUAL REVIEW REPORT (LOD2)
        Institution : LCH Ltd
        Période d'analyse : Q4-2025 (Comparaison YoY vs Q2-2025)
        
        [1. COUNTERPARTY CREDIT RATING & SCORING]
        - Internal Rating Assigné : AA
        - Score de Risque Global   : 67.71 / 100
        - Probabilité de Défaut (PD) : 3.0%

        [2. LIMIT SETTING & EXPOSURE THRESHOLDS]
        - Limite Activité Clearing (Default Fund Sizing) : 6194.71 M$
        - Limite Activité Bilatérale (FMI Directe)         : 7186.06 M$

        [3. QUANTITATIVE DISCLOSURES & TREND ANALYSIS (YoY)]
        - Default Fund Size Actuel : 8849.58 M$ (Évolution YoY : -2.67%)
        - Liquid Resources Available: 20531.6 M$
        - Peak Liquidity Need       : 16269.8 M$
        - Liquidity Coverage Ratio  : 1.26x

        [4. AUTOMATED RISK COMMENTARY & ESCALATION]
        
[STATUT STABLE] Le profil financier de LCH Ltd est robuste. Les métriques de liquidité et de cascade de défaut respecten